In [12]:
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [14]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [15]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the database for information about playlists"""
    
    try:
        return db.run(query)
    except Exception as e:
        return f"An error occurred while querying the database: {e}"

Create State:

In [16]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

Create SubAgent:

In [17]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pprint import pprint
from langgraph.checkpoint.memory import InMemorySaver

travel_agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="You are a travel agent. No follow up questions."
)

venue_agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt="You are a wedding venue agent. No follow up questions."
)

playlist_agent = create_agent(
    model="gpt-5-nano",
    tools=[query_playlist_db],
    checkpointer=InMemorySaver(),
    system_prompt="You are a wedding playlist agent who can query a database for information about playlists. No follow up questions."
)

In [18]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]})
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )

## Creating the main agent

coord_agent = create_agent(
    model='gpt-5-nano',
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="You are a helpful assistant who can call subagents to help with wedding planning. " \
    "You can call the travel agent, venue agent, and playlist agent to get information about flights, " \
    "venues, and playlists respectively. No follow up questions.")

In [19]:
from langchain.messages import HumanMessage

question = "I am planning a wedding and I need to find a venue, book flights for my guests, and create a playlist for the reception. Can you help me with that?"

response = coord_agent.invoke({"messages": [HumanMessage(content=question)]})

KeyError: 'genre'